# MedSafe-AI · Component 3 · Full pipeline in Google Colab
**Drug-pair side-effect prediction** (XGBoost, Random Forest, SVM) - the whole job in one notebook.

**What it does (top to bottom)**
1. Gets your code and small data files from **GitHub** (one repo, see `PUSH_TO_GITHUB.md`).
2. Finds each drug's chemical structure on PubChem and makes **fingerprints** with RDKit (RDKit works in Colab).
3. Builds the final table, makes the **70 / 20 / 10 split** and checks it (no leakage).
4. Trains **Random Forest, XGBoost, SVM** and compares them with two simple baselines.
5. Saves the results (download as a zip, and optionally push them back to GitHub).

**Before you start**
* *Runtime → Change runtime type → **T4 GPU*** (this makes XGBoost faster; CPU also works, just slower).
* The repo is set to `sksivakajan/mac-doc`. If it is **private** (set `PRIVATE = True` in cell A1): left side **key icon (Secrets)** → *Add new secret* → name `GITHUB_TOKEN` → paste a token → switch **Notebook access ON**. The token is never printed.
* **File → Save a copy in Drive** first. Then your printed tables and plots are kept inside the notebook too.
* Cell A2 asks you once to allow Google Drive. Everything (fingerprints, split, **every trained model**, metrics, plots) is stored in your Drive folder `MedSafe_C3_store`; an interrupted session continues where it stopped (finished models are loaded, not trained again).
* Then **Runtime → Run all**. Total time: about 30-60 minutes the first time.

## Part A · Get the code from GitHub

In [ ]:
#@title A1 · Connect to GitHub and download the code
import os, re, subprocess

REPO = "sksivakajan/mac-doc"          # GitHub  owner/repo  (or a full https URL)
BRANCH = "main"
SUBFOLDER = "component3"   # folder inside the repo that holds c3_utils.py ("" if it is at the top)
PRIVATE = False      # False = public repo.  True = private repo (needs the GITHUB_TOKEN secret).

TOKEN = ""
if PRIVATE:
    try:
        from google.colab import userdata
        TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        raise RuntimeError("Private repo, but the secret GITHUB_TOKEN was not found. Add it under the key icon "
                           "(Secrets) and switch 'Notebook access' on.") from e

clean_url = REPO if "://" in REPO or REPO.startswith("/") else f"https://github.com/{REPO}.git"
url = clean_url.replace("https://", f"https://x-access-token:{TOKEN}@") if TOKEN and clean_url.startswith("https://") else clean_url

def run(cmd, **kw):
    """Run a command; never show the token."""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    out = (r.stdout + r.stderr).replace(TOKEN, "***") if TOKEN else (r.stdout + r.stderr)
    if r.returncode:
        raise RuntimeError("command failed: " + " ".join(cmd[:3]) + "\n" + out[-800:])
    return out

REPO_DIR = "/content/repo"
if os.path.isdir(REPO_DIR):
    run(["rm", "-rf", REPO_DIR])
run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR])
run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clean_url])      # do not keep the token in .git/config
PROJECT = os.path.join(REPO_DIR, SUBFOLDER) if SUBFOLDER else REPO_DIR
os.chdir(PROJECT)
print("Working folder:", os.getcwd())
print("Files:", sorted(os.listdir(".")))
assert os.path.exists("c3_utils.py"), "c3_utils.py not found here. Check SUBFOLDER."
assert os.path.exists("work/pairs_raw.csv"), "work/pairs_raw.csv not found. Did you push the small work files? (see PUSH_TO_GITHUB.md)"

In [ ]:
#@title A2 · Keep everything on Google Drive (so nothing is lost when Colab stops)
# Colab asks you ONCE to allow Google Drive. Click Connect / Allow yourself.
from google.colab import drive
import shutil, os
from pathlib import Path
# If a pop-up window appears, click "Connect to Google Drive", choose your account, then press "Continue" / "Allow" on EVERY page.
# Pop-ups must not be blocked in your browser. If it fails, we use a local folder instead (no Drive) so you can still go on.
DRIVE_OK = False
for _try in range(2):
    try:
        drive.mount("/content/drive", force_remount=True)
        DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
        if DRIVE_OK:
            break
    except Exception as e:
        print(f"Drive mount failed (try {_try+1}):", str(e)[:120])
if DRIVE_OK:
    STORE = Path("/content/drive/MyDrive/MedSafe_C3_store")
else:
    STORE = Path("/content/MedSafe_C3_store")
    print("\nWARNING: Google Drive is NOT connected. Files are kept only in this Colab session.")
    print("Run Part F1 at the end to download a zip, or fix Drive and run this cell again.")
(STORE / "results").mkdir(parents=True, exist_ok=True)
(STORE / "work").mkdir(parents=True, exist_ok=True)

def save_all(note=""):
    """Copy the work folder to Google Drive (the results folder IS on Drive already, see below)."""
    shutil.copytree("work", STORE / "work", dirs_exist_ok=True)
    print(("saved to Google Drive:" if DRIVE_OK else "saved locally (no Drive):"), STORE, note)

# bring back what an earlier session already made (fingerprints, split, trained models ...)
shutil.copytree(STORE / "work", "work", dirs_exist_ok=True)
if Path("results").exists() and not Path("results").is_symlink():
    shutil.copytree("results", STORE / "results", dirs_exist_ok=True)
    shutil.rmtree("results")
if not Path("results").exists():
    os.symlink(STORE / "results", "results", target_is_directory=True)     # every file written to results/ goes straight to Drive
print("Earlier files found on Drive:", sorted(p.name for p in (STORE / "results").iterdir())[:20])

## Part B · Install the packages

In [ ]:
#@title B1 · Install (about 1 minute)
%pip -q install rdkit xgboost scikit-learn pandas numpy matplotlib joblib
import xgboost, sklearn, rdkit
print("xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__, "| RDKit", rdkit.__version__)
import shutil
print("GPU found:", bool(shutil.which("nvidia-smi")))

## Part C · Drug details, structures and fingerprints (notebook 02)
The fast PubChem + RDKit step runs first (about 10 minutes). The next cells then use its result.

In [ ]:
import sys, importlib.util, subprocess
from pathlib import Path

HERE = Path.cwd()                                   # the folder of this notebook (component3)
if not (HERE / "c3_utils.py").exists():
    if Path("/content").exists():                   # Google Colab: get the code from GitHub
        REPO_DIR = Path("/content/repo")
        if not (REPO_DIR / "component3" / "c3_utils.py").exists():
            subprocess.run(["rm", "-rf", str(REPO_DIR)])
            subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/sksivakajan/mac-doc.git", str(REPO_DIR)])
        HERE = REPO_DIR / "component3"
        import os; os.chdir(HERE)
        print("Colab: code taken from GitHub ->", HERE)
    else:                                           # Jupyter was started from another folder
        HERE = Path(r"D:\med\data set\component3")
sys.path.insert(0, str(HERE))

def need(*pairs):
    """Install the packages that are missing. pairs = (import name, pip name)."""
    missing = [pip for mod, pip in pairs if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", missing)
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as e:
            print("!! Could not install", missing, "->", e)

need(("pandas", "pandas"), ("numpy", "numpy"), ("rdkit", "rdkit"))

import time
import numpy as np, pandas as pd
import c3_utils as c3

try:
    DATA_DIR = c3.find_data_dir()
except FileNotFoundError:
    DATA_DIR = None                       # e.g. in Google Colab: the saved drug_details_raw.csv is used instead
WORK = HERE / "work"
LABEL_FILES = sorted(DATA_DIR.glob("drug-label-*.json")) if DATA_DIR else []
print("Label files found:", len(LABEL_FILES), "(14 on your PC; 0 is fine in Colab)")
print("RDKit available  :", c3.have_rdkit())
print("Python in use     :", sys.executable)

## Step 1 · Settings

In [ ]:
N_BITS = 1024          # length of the fingerprint of one drug
RADIUS = 2             # Morgan radius (2 is the normal choice)
TOP_CLASSES = 100      # how many drug classes to use as features
MAX_LABELS = None      # set to e.g. 20000 for a quick test run; None = read everything

## Step 2 · Drug class, use and label side effects (openFDA labels)
Each label file is about 600 MB, so they are read as a **stream** (not loaded all at once).
Only labels with **one** active ingredient are used, so the class belongs clearly to that drug.

In [ ]:
drugs = pd.read_csv(WORK / "drugs_raw.csv")

# names we look for: exact name first, then the name without salt words (citrate, sodium ...)
wanted = {}
for did, nm in zip(drugs["drug_id"], drugs["drug_name"]):
    wanted.setdefault(c3.norm_name(nm), did)
for did, nm in zip(drugs["drug_id"], drugs["drug_name"]):
    for v in c3.name_variants(nm):
        wanted.setdefault(v, did)

DETAILS_FILE = WORK / "drug_details_raw.csv"
if DETAILS_FILE.exists() and MAX_LABELS is None:
    details = pd.read_csv(DETAILS_FILE, keep_default_na=False)
    print("Loaded saved details:", DETAILS_FILE.name)
else:
    if not LABEL_FILES:
        raise FileNotFoundError("No openFDA label files and no saved drug_details_raw.csv in the work folder.")
    t0 = time.time()
    details = c3.scan_labels(LABEL_FILES, wanted, max_labels=MAX_LABELS)
    if MAX_LABELS is None:
        details.to_csv(DETAILS_FILE, index=False)
    print(f"Done in {(time.time()-t0)/60:.1f} minutes")

drugs = drugs.merge(details, on="drug_id", how="left").fillna({"drug_class": "", "all_classes": "", "main_use": "",
                                                               "label_adverse_reactions": "", "n_labels": 0})
print(f"drugs with a label match : {(drugs['n_labels'] > 0).sum():,} of {len(drugs):,}")
print(f"drugs with a drug class  : {(drugs['drug_class'] != '').sum():,}")
print(f"drugs with a main use    : {(drugs['main_use'] != '').sum():,}")
display(drugs[drugs["drug_class"] != ""][["drug_name", "drug_class", "main_use"]].head(8))

## Step 2b · (Colab) Fast structures + fingerprints
Uses PubChem with 4 parallel workers and RDKit. Skipped if `work/drug_fp_colab.npz` already exists.

In [ ]:
import colab_fp
colab_fp.build_from_work("work")

## Step 3 · Chemical structure (SMILES) from PubChem
PubChem allows about 5 requests per second, so ~1,800 drugs take around 10 minutes. If PubChem is busy, the cell waits and tries again. Temporary failures are **never** saved as "not found"; if some remain, just run the cell again.
Only drugs PubChem really does not know (for example mixtures such as *Coal Tar*) are listed and **left out** of the models (we can not make a fingerprint for them).

In [ ]:
FP_FILE = WORK / "drug_fp_colab.npz"      # made in Google Colab: lets a PC without RDKit go on
if FP_FILE.exists():
    z = np.load(FP_FILE, allow_pickle=False)
    assert int(z["n_bits"]) == N_BITS and int(z["radius"]) == RADIUS, "the Colab fingerprint file has other settings (N_BITS / RADIUS)"
    drugs["smiles"] = drugs["drug_id"].map(dict(zip(z["ids"].tolist(), z["smiles"].tolist()))).fillna("")
    drugs["smiles_status"] = drugs["drug_id"].map(dict(zip(z["ids"].tolist(), z["status"].tolist()))).fillna("unknown")
    print("Using the structures and fingerprints made in Colab:", FP_FILE.name)
else:
    drugs = c3.get_smiles(drugs, WORK / "smiles_cache.csv")
print(drugs["smiles_status"].value_counts().to_dict())
have = drugs["smiles"] != ""
print(f"SMILES found: {have.sum():,} of {len(drugs):,} ({100*have.mean():.1f}%)")
drugs[~have][["drug_id", "drug_name", "smiles_status"]].to_csv(WORK / "drugs_without_smiles.csv", index=False)
print("Examples PubChem does not know:", list(drugs[drugs["smiles_status"] == "not_found"]["drug_name"].head(12)))

unknown = (drugs["smiles_status"] == "unknown").sum()
if unknown:
    raise RuntimeError(f"{unknown} drugs could not be checked (PubChem busy or internet problem). "
                       "Run THIS cell again: it only asks PubChem for the missing ones. Do not continue before this is 0.")

## Step 4 · Fingerprints
A Morgan fingerprint is a 0/1 list that describes which small chemical pieces a drug contains.
Drugs with similar structure get similar fingerprints, so the model can guess for drugs it has seen little.

In [ ]:
if FP_FILE.exists():
    idmap = {i: k for k, i in enumerate(z["ids"].tolist())}
    drugs["has_fp"] = [(i in idmap) and bool(z["valid"][idmap[i]]) for i in drugs["drug_id"]]
    print(f"valid fingerprints: {int(drugs['has_fp'].sum()):,} of {len(drugs):,}")
    drugs_final = drugs[drugs["has_fp"]].reset_index(drop=True)
    FP = z["fp"][[idmap[i] for i in drugs_final["drug_id"]]].astype(np.uint8)
    print("fingerprint matrix:", FP.shape, "  average bits set per drug:", round(float(FP.sum(1).mean()), 1))
else:
    if not c3.have_rdkit():
        err = c3.rdkit_error()
        print("Python in use:", sys.version.split()[0], "-", sys.executable)
        print("RDKit problem:", err)
        if "Application Control" in err or "blocked" in err.lower():
            raise RuntimeError(
                "Windows is BLOCKING RDKit on this computer (Application Control / Smart App Control). "
                "Do not try to work around it. Use the notebook  02b_fingerprints_colab.ipynb  in Google Colab "
                "(instructions are inside it), unzip its result into the work folder, then go on with 03.")
        print("Trying to install RDKit now ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "rdkit"])
        except Exception as e:
            print("pip install failed:", e)
    if not c3.have_rdkit():
        raise RuntimeError(
            "RDKit is still not available: " + c3.rdkit_error() + "\n"
            "Try in a new cell:   %pip install rdkit   then run this cell again.\n"
            "If it still fails use  02b_fingerprints_colab.ipynb  in Google Colab.")

    fps, ok = [], []
    for smi in drugs["smiles"]:
        fp = c3.morgan_fp(smi, RADIUS, N_BITS) if smi else None
        ok.append(fp is not None)
        fps.append(fp)
    drugs["has_fp"] = ok
    print(f"valid fingerprints: {sum(ok):,} of {len(drugs):,}")

    drugs_final = drugs[drugs["has_fp"]].reset_index(drop=True)
    FP = np.stack([f for f in fps if f is not None]).astype(np.uint8)
    print("fingerprint matrix:", FP.shape, "  average bits set per drug:", round(float(FP.sum(1).mean()), 1))

## Step 5 · Drug classes as numbers

In [ ]:
class_lists = [[c for c in str(s).split("; ") if c] for s in drugs_final["all_classes"]]
CM, class_names = c3.class_matrix(class_lists, TOP_CLASSES)
print("class matrix:", CM.shape, "  drugs with at least one class:", int((CM.sum(1) > 0).sum()))
print("most common classes:", class_names[:8])

## Step 6 · Final pair table for the models
We keep only pairs where **both** drugs have a fingerprint. Pairs are stored with the **row number** of each drug (`idx_a`, `idx_b`).

In [ ]:
pairs = pd.read_csv(WORK / "pairs_raw.csv")
Y = np.load(WORK / "Y_raw.npy")
effects = pd.read_csv(WORK / "effects.csv")

row_of = {int(d): i for i, d in enumerate(drugs_final["drug_id"])}
ia = pairs["drug_a_id"].map(row_of)
ib = pairs["drug_b_id"].map(row_of)
keep = ia.notna() & ib.notna()
print(f"pairs before: {len(pairs):,}   pairs with both drugs usable: {keep.sum():,} ({100*keep.mean():.1f}%)")

pairs_f = pairs[keep].copy()
pairs_f["idx_a"] = ia[keep].astype(int).values
pairs_f["idx_b"] = ib[keep].astype(int).values
Y_f = Y[keep.values]

# an effect that lost every positive can not be learned -> report it
zero = effects[Y_f.sum(0) == 0]["effect_name"].tolist()
print("side effects with no positive pair left:", zero if zero else "none")

pairs_f.reset_index(drop=True).to_csv(WORK / "modeling_pairs.csv", index=False)
np.save(WORK / "modeling_Y.npy", Y_f)
np.savez_compressed(WORK / "drug_features.npz", FP=FP, CM=CM, class_names=np.array(class_names))
drugs_final.drop(columns=["has_fp"]).to_csv(WORK / "drugs_final.csv", index=False)
effects.to_csv(WORK / "modeling_effects.csv", index=False)
print(f"saved: {len(pairs_f):,} pairs, {len(drugs_final):,} drugs, {Y_f.shape[1]} side effects")

In [ ]:
save_all('after part C')

## Part D · 70 / 20 / 10 split and data checks (notebook 03)

In [ ]:
import sys, importlib.util, subprocess
from pathlib import Path

HERE = Path.cwd()                                   # the folder of this notebook (component3)
if not (HERE / "c3_utils.py").exists():
    if Path("/content").exists():                   # Google Colab: get the code from GitHub
        REPO_DIR = Path("/content/repo")
        if not (REPO_DIR / "component3" / "c3_utils.py").exists():
            subprocess.run(["rm", "-rf", str(REPO_DIR)])
            subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/sksivakajan/mac-doc.git", str(REPO_DIR)])
        HERE = REPO_DIR / "component3"
        import os; os.chdir(HERE)
        print("Colab: code taken from GitHub ->", HERE)
    else:                                           # Jupyter was started from another folder
        HERE = Path(r"D:\med\data set\component3")
sys.path.insert(0, str(HERE))

def need(*pairs):
    """Install the packages that are missing. pairs = (import name, pip name)."""
    missing = [pip for mod, pip in pairs if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", missing)
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as e:
            print("!! Could not install", missing, "->", e)

need(("pandas", "pandas"), ("numpy", "numpy"), ("matplotlib", "matplotlib"))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import c3_utils as c3

WORK = HERE / "work"
pairs = pd.read_csv(WORK / "modeling_pairs.csv")
Y = np.load(WORK / "modeling_Y.npy")
effects = pd.read_csv(WORK / "modeling_effects.csv")
ia, ib = pairs["idx_a"].to_numpy(), pairs["idx_b"].to_numpy()
print(f"{len(pairs):,} drug pairs, {Y.shape[1]} side effects, {len(set(ia) | set(ib)):,} drugs")

## Step 1 · Make the splits

In [ ]:
FRACS = (0.70, 0.20, 0.10)
SEED = c3.SEED

splits = {
    "pair": c3.split_random(len(pairs), FRACS, SEED),
    "unseen_drug": c3.split_unseen_drug(ia, ib, FRACS, SEED),
}
for mode, (tr, va, te) in splits.items():
    n = len(pairs)
    print(f"{mode:12s} train {len(tr):>7,} ({100*len(tr)/n:4.1f}%)   val {len(va):>7,} ({100*len(va)/n:4.1f}%)   test {len(te):>7,} ({100*len(te)/n:4.1f}%)")
    np.savez(WORK / f"split_{mode}.npz", train=tr, val=va, test=te)

## Step 2 · Check the data after the split
All of these must look right **before** you train anything.

In [ ]:
reports = {}
for mode, parts in splits.items():
    rep = c3.check_split(ia, ib, Y, parts, effects)
    reports[mode] = rep
    print("=" * 70)
    print("SPLIT:", mode)
    print("-- 1. Same pair in two parts? (must all be 0)")
    print("  ", rep["pair_overlap"])
    print("-- 2. Duplicate pairs in the whole table (must be 0):", rep["duplicate_pairs_total"])
    print("-- 3. Drugs that were never in TRAINING")
    display(pd.DataFrame(rep["drug_overlap"]).set_index("part"))
    print("-- 4. Labels in each part")
    display(pd.DataFrame(rep["labels"]).set_index("part"))
    for part, names in rep["zero_pos_effects"].items():
        if names:
            print(f"  !! side effects with NO positive pair in {part}: {names[:10]}")
    assert all(v == 0 for v in rep["pair_overlap"].values()), "A pair is in two parts!"
    assert rep["duplicate_pairs_total"] == 0, "Duplicate pairs found!"
print("\nAll hard checks passed.")

### How to read the checks
* **Check 1 and 2** must be zero. If not, the test score would be too good (the model saw the answer).
* **Check 3**: in the `pair` split a few validation/test pairs may contain a drug that is not in training. In the `unseen_drug` split this is 100 % by design. Report **both** results in the thesis.
* **Check 4**: the share of positives should be similar in all parts. In the `unseen_drug` split it can differ a little because different drugs have different side-effect profiles.

In [ ]:
# Easy baseline: "always predict the most common side effects". Any model must beat this.
for mode, (tr, va, te) in splits.items():
    prev = Y[tr].mean(0)
    S = np.tile(prev, (len(te), 1))
    r = c3.evaluate(Y[te], S, c3.best_threshold(Y[va], np.tile(prev, (len(va), 1))))
    print(f"{mode:12s} baseline on TEST:  micro-AUPRC {r['micro_AUPRC']:.3f}   P@3 {r['P@3']:.3f}   P@10 {r['P@10']:.3f}")

# plot: how common is each side effect in train / val / test (pair split)
tr, va, te = splits["pair"]
fig, ax = plt.subplots(figsize=(11, 3.2))
x = np.arange(Y.shape[1])
for ix, nm, col in [(tr, "train", "#3b6ea5"), (va, "val", "#e0a030"), (te, "test", "#c0504d")]:
    ax.plot(x, Y[ix].mean(0), label=nm, color=col, lw=1.4)
ax.set_xlabel("side effect (most common first)"); ax.set_ylabel("share of pairs"); ax.legend(); ax.set_title("Side-effect frequency per part (pair split)")
plt.tight_layout(); plt.show()

In [ ]:
save_all('after part D')

## Part E1 · Train and compare the models - random pair split (notebook 04)

In [ ]:
import sys, importlib.util, subprocess
from pathlib import Path

HERE = Path.cwd()                                   # the folder of this notebook (component3)
if not (HERE / "c3_utils.py").exists():
    if Path("/content").exists():                   # Google Colab: get the code from GitHub
        REPO_DIR = Path("/content/repo")
        if not (REPO_DIR / "component3" / "c3_utils.py").exists():
            subprocess.run(["rm", "-rf", str(REPO_DIR)])
            subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/sksivakajan/mac-doc.git", str(REPO_DIR)])
        HERE = REPO_DIR / "component3"
        import os; os.chdir(HERE)
        print("Colab: code taken from GitHub ->", HERE)
    else:                                           # Jupyter was started from another folder
        HERE = Path(r"D:\med\data set\component3")
sys.path.insert(0, str(HERE))

def need(*pairs):
    """Install the packages that are missing. pairs = (import name, pip name)."""
    missing = [pip for mod, pip in pairs if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", missing)
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as e:
            print("!! Could not install", missing, "->", e)

need(("pandas", "pandas"), ("numpy", "numpy"), ("matplotlib", "matplotlib"),
     ("sklearn", "scikit-learn"), ("xgboost", "xgboost"), ("joblib", "joblib"))

import os, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
import c3_utils as c3
warnings.filterwarnings("ignore")

WORK = HERE / "work"
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)
SEED = c3.SEED

## Step 1 · Settings
| Setting | Meaning |
|---|---|
| `SPLIT_MODE` | `"pair"` (random pairs) or `"unseen_drug"` (drugs held out). |
| `REUSE_SAVED` | `True` = load a saved model (same settings) instead of training again. Every trained model is saved in `results/`. |
| `XGB_DEVICE` | `"cpu"` or `"cuda"` (GPU, Colab only). |
| `TRAIN_CAP` | Use at most this many training pairs. Smaller = faster. Set `None` to use all (needs more RAM and hours of time). |
| `SVM_TRAIN_CAP` | The SVM is slow, so it uses fewer pairs. |
| `USE_DRUG_HISTORY` | Add "how often did each drug cause each side effect in training" as features. |
| `TUNE` | `True` tries 2 settings per model on a small sample and keeps the better one (slower). |

In [ ]:
SPLIT_MODE = "pair"            # "pair"  or  "unseen_drug"
TRAIN_CAP = 60_000
SVM_TRAIN_CAP = 30_000
USE_DRUG_HISTORY = True
TUNE = False
MODELS_TO_RUN = ["random_forest", "xgboost", "svm"]
SVM_KERNEL = "linear"          # "linear" (fast) or "rbf" (very slow; use SVM_TRAIN_CAP <= 10_000)
N_JOBS = os.cpu_count() or 1
XGB_DEVICE = "cuda" if __import__("shutil").which("nvidia-smi") else "cpu"             # "cuda" = use a GPU (Google Colab with a GPU runtime makes XGBoost much faster)
REUSE_SAVED = True            # a model trained before with the SAME settings is loaded from the results folder instead of trained again

PARAMS = {   # first entry is used when TUNE = False
    "random_forest": [dict(n_estimators=150, min_samples_leaf=5, max_features="sqrt"),
                      dict(n_estimators=150, min_samples_leaf=2, max_features="sqrt")],
    "xgboost":       [dict(n_estimators=150, max_depth=5, learning_rate=0.1),
                      dict(n_estimators=150, max_depth=7, learning_rate=0.05)],
    "svm":           [dict(C=0.05), dict(C=0.5)],
}
print("split:", SPLIT_MODE, "| cores:", N_JOBS)

## Step 2 · Load the data and the split

In [ ]:
pairs = pd.read_csv(WORK / "modeling_pairs.csv")
Y = np.load(WORK / "modeling_Y.npy")
effects = pd.read_csv(WORK / "modeling_effects.csv")
drugs = pd.read_csv(WORK / "drugs_final.csv", keep_default_na=False)
z = np.load(WORK / "drug_features.npz")
FP, CM = z["FP"], z["CM"]
sp = np.load(WORK / f"split_{SPLIT_MODE}.npz")
tr_all, va, te = sp["train"], sp["val"], sp["test"]
ia, ib = pairs["idx_a"].to_numpy(), pairs["idx_b"].to_numpy()

sev_rank = effects["severity"].map({"Low": 0, "Medium": 1, "High": 2}).fillna(-1).to_numpy().astype(int)
high_mask = sev_rank == 2
print(f"train {len(tr_all):,}   val {len(va):,}   test {len(te):,}   | side effects {Y.shape[1]} (High severity: {high_mask.sum()})")

rng = np.random.default_rng(SEED)
def cap(idx, n):
    return np.sort(rng.choice(idx, n, replace=False)) if (n and len(idx) > n) else idx
tr = cap(tr_all, TRAIN_CAP)
print(f"pairs used for training: {len(tr):,}")

## Step 3 · Build the features
For a pair (A, B) the input numbers are (the same for A+B and B+A):
* **Fingerprints:** `A OR B` and `A AND B` (2 × 1024 numbers).
* **Drug classes:** which classes either drug has, and which both have.
* **Drug history** (optional): for each side effect, how often drug A and drug B had it in **training pairs**. For a *training* row the pair's own answer is removed first (leave-one-out), so the model can not read its own label. Validation/test rows never take part in this.

In [ ]:
hist = c3.DrugHistory(n_drugs=len(drugs)).fit(ia[tr_all], ib[tr_all], Y[tr_all]) if USE_DRUG_HISTORY else None

def build_X(idx, is_train):
    a, b = ia[idx], ib[idx]
    parts = [c3.pair_fp_matrix(a, b, FP), CM[a] | CM[b], CM[a] & CM[b]]
    X = np.hstack(parts).astype(np.float32)
    if hist is not None:
        X = np.hstack([X, hist.transform(a, b, Y[idx] if is_train else None)])
    return X

t0 = time.time()
Xtr, Xva, Xte = build_X(tr, True), build_X(va, False), build_X(te, False)
Ytr, Yva, Yte = Y[tr], Y[va], Y[te]
print(f"features: {Xtr.shape[1]} per pair   | train {Xtr.shape}  val {Xva.shape}  test {Xte.shape}   ({time.time()-t0:.0f}s)")
print(f"memory: about {(Xtr.nbytes + Xva.nbytes + Xte.nbytes)/1e9:.1f} GB")

## Step 4 · Baselines (no machine learning)
* **prevalence** – every pair gets the same list: the most common side effects in training.
* **drug_history** – average of the two drugs' own history (needs no model, no fingerprints).

If a trained model can not beat these, it has not learned anything useful.

In [ ]:
results, val_scores, test_scores = [], {}, {}

def record(name, S_va, S_te, extra=None):
    thr = c3.best_threshold(Yva, S_va)                 # cut-off chosen on VALIDATION only
    rv = c3.evaluate(Yva, S_va, thr, high_mask)
    rt = c3.evaluate(Yte, S_te, thr, high_mask)
    row = {"model": name, "threshold": round(thr, 4), **{f"val_{k}": v for k, v in rv.items()},
           **{f"test_{k}": v for k, v in rt.items()}}
    if extra:
        row.update(extra)
    results.append(row)
    val_scores[name], test_scores[name] = S_va, S_te
    print(f"{name:16s} val micro-AUPRC {rv['micro_AUPRC']:.3f} | TEST micro-AUPRC {rt['micro_AUPRC']:.3f}  "
          f"P@3 {rt['P@3']:.3f}  P@10 {rt['P@10']:.3f}  micro-F1 {rt['micro_F1']:.3f}  High-recall {rt.get('recall_High_effects', float('nan')):.3f}")

base = c3.PrevalenceBaseline().fit(Xtr, Y[tr_all])
record("prevalence", base.predict_proba(Xva), base.predict_proba(Xte))

if hist is not None:
    K = Y.shape[1]
    def hist_score(idx):
        h = hist.transform(ia[idx], ib[idx], None)
        return (h[:, :K] + h[:, K:2 * K]) / 2
    record("drug_history", hist_score(va), hist_score(te))

## Step 5 · Train the three models
Time is printed for each one. Run this cell once; the models stay in memory for the next steps.

In [ ]:
def make_model(name, p):
    from sklearn.multiclass import OneVsRestClassifier
    if name == "random_forest":
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(n_jobs=N_JOBS, random_state=SEED, **p)
    if name == "xgboost":
        from xgboost import XGBClassifier
        return OneVsRestClassifier(XGBClassifier(tree_method="hist", device=XGB_DEVICE, subsample=0.8, colsample_bytree=0.5,
                                                 n_jobs=N_JOBS, eval_metric="logloss", random_state=SEED,
                                                 verbosity=0, **p))
    if name == "svm":
        if SVM_KERNEL == "linear":
            from sklearn.svm import LinearSVC
            base = LinearSVC(dual=False, random_state=SEED, **p)
        else:
            from sklearn.svm import SVC
            base = SVC(kernel="rbf", gamma="scale", **p)
        return OneVsRestClassifier(base, n_jobs=N_JOBS)
    raise ValueError(name)

trained = {}
for name in MODELS_TO_RUN:
    n_fit = min(len(Xtr), SVM_TRAIN_CAP) if name == "svm" else len(Xtr)
    Xf, Yf = Xtr[:n_fit], Ytr[:n_fit]
    params = PARAMS[name][0]
    t0 = time.time()
    if TUNE:                                   # pick the better setting on VALIDATION, using a small sample
        n_small = min(15_000, n_fit)
        best_p, best_s = None, -1
        for p in PARAMS[name]:
            m = make_model(name, p).fit(Xf[:n_small], Yf[:n_small])
            s = c3.evaluate(Yva, c3.predict_scores(m, Xva), 0.5)["micro_AUPRC"]
            print(f"   {name} {p} -> val micro-AUPRC {s:.3f}")
            if s > best_s:
                best_p, best_s = p, s
        params = best_p
    sig = dict(name=name, params=str(params), n_fit=int(n_fit), mode=SPLIT_MODE, drug_history=bool(USE_DRUG_HISTORY),
               svm_kernel=SVM_KERNEL if name == "svm" else "", n_features=int(Xf.shape[1]))
    mfile = RESULTS / f"model_{SPLIT_MODE}_{name}.joblib"
    model, secs = None, 0.0
    if REUSE_SAVED and mfile.exists():
        saved = joblib.load(mfile)
        if saved.get("sig") == sig:
            model, secs = saved["model"], saved["seconds"]
            print(f"   loaded the saved {name} model ({mfile.name}); it was trained in {secs/60:.1f} min")
    if model is None:
        model = make_model(name, params).fit(Xf, Yf)
        secs = time.time() - t0
        joblib.dump({"model": model, "sig": sig, "seconds": secs}, mfile, compress=3)   # keep it: no need to train again
    trained[name] = model
    record(name, c3.predict_scores(model, Xva), c3.predict_scores(model, Xte),
           {"params": str(params), "train_pairs": n_fit, "train_seconds": round(secs)})
    print(f"   -> trained in {secs/60:.1f} min\n")

## Step 6 · Results table

In [ ]:
res = pd.DataFrame(results)
cols = ["model", "test_micro_AUPRC", "test_macro_AUPRC", "test_P@3", "test_P@5", "test_P@10", "test_R@10",
        "test_micro_F1", "test_macro_F1", "test_recall_High_effects", "val_micro_AUPRC"]
show = res[cols].round(3)
display(show)
res.to_csv(RESULTS / f"metrics_{SPLIT_MODE}.csv", index=False)

learned = res[~res["model"].isin(["prevalence", "drug_history"])]
best_name = learned.sort_values("val_micro_AUPRC", ascending=False)["model"].iloc[0]   # chosen on VALIDATION
print("Best trained model (by validation micro-AUPRC):", best_name)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, m, t in zip(ax, ["test_micro_AUPRC", "test_P@3", "test_micro_F1"], ["micro-AUPRC", "Precision@3", "micro-F1"]):
    a.bar(res["model"], res[m], color=["#999" if x in ("prevalence", "drug_history") else "#3b6ea5" for x in res["model"]])
    a.set_title(t + " (test)"); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(RESULTS / f"comparison_{SPLIT_MODE}.png", dpi=130); plt.show()

## Step 7 · Sanity checks (do this before you trust the numbers)
1. **Overfitting gap:** training score far above test score means the model memorised.
2. **Label-shuffle test:** we shuffle the training answers and train again. A healthy pipeline must then fall back to the *prevalence* baseline. If it stays clearly **above** the baseline, labels are leaking into the features.

In [ ]:
# 1. overfitting gap on a sample of training rows
sample = np.arange(min(8000, len(Xtr)))
rows = []
for name, m in trained.items():
    s_tr = c3.evaluate(Ytr[sample], c3.predict_scores(m, Xtr[sample]), 0.5)["micro_AUPRC"]
    s_te = res.loc[res["model"] == name, "test_micro_AUPRC"].iloc[0]
    rows.append({"model": name, "train_micro_AUPRC": round(s_tr, 3), "test_micro_AUPRC": round(s_te, 3), "gap": round(s_tr - s_te, 3)})
display(pd.DataFrame(rows))
print("Note: a gap is normal for Random Forest (it sees its own training rows). Look at the TEST column for quality.")

# 2. label-shuffle test (fast Random Forest on a small sample)
from sklearn.ensemble import RandomForestClassifier
Ys = Y[tr_all].copy()
np.random.default_rng(1).shuffle(Ys, axis=0)                    # break the link between pairs and answers
hist_s = c3.DrugHistory(len(drugs)).fit(ia[tr_all], ib[tr_all], Ys) if USE_DRUG_HISTORY else None
sub = tr[:15_000]
pos = {int(p): i for i, p in enumerate(tr_all)}
Ys_sub = Ys[[pos[int(p)] for p in sub]]
def X_shuf(idx, own=None):
    a, b = ia[idx], ib[idx]
    X = np.hstack([c3.pair_fp_matrix(a, b, FP), CM[a] | CM[b], CM[a] & CM[b]]).astype(np.float32)
    return np.hstack([X, hist_s.transform(a, b, own)]) if hist_s is not None else X
rf = RandomForestClassifier(n_estimators=60, min_samples_leaf=5, n_jobs=N_JOBS, random_state=SEED).fit(X_shuf(sub, Ys_sub), Ys_sub)
shuf = c3.evaluate(Yte, c3.predict_scores(rf, X_shuf(te)), 0.5)["micro_AUPRC"]
basel = res.loc[res["model"] == "prevalence", "test_micro_AUPRC"].iloc[0]
print(f"label-shuffle model: test micro-AUPRC {shuf:.3f}   |   prevalence baseline {basel:.3f}")
print("PASS: no sign of leakage" if shuf < basel + 0.02 else "!! WARNING: shuffled model beats the baseline -> look for leakage")

## Step 8 · Severity (Low / Medium / High) for a pair
A pair gets the **worst severity among its side effects**. We compare the true severity (from the TWOSIDES signals) with the severity from the predicted side effects.

⚠ The severity of each side effect comes from the placeholder rules in `severity.py`. Agree a real source with your supervisor before using these numbers in the thesis.

⚠ About 28 of the side effects are marked High, so **most pairs contain at least one High effect**. "Worst severity" is therefore dominated by the High class. Talk with your supervisor about a better pair-level rule, for example *"High only if the predicted chance of a High effect is above a higher cut-off"*, or a severity **score** (the sum of the probabilities of High effects).

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score
thr = float(res.loc[res["model"] == best_name, "threshold"].iloc[0])
S_best = test_scores[best_name]
true_sev = c3.pair_severity(Yte, sev_rank)
pred_sev = c3.pair_severity(S_best >= thr, sev_rank)
ok = true_sev >= 0
labels = [0, 1, 2]
cm = confusion_matrix(true_sev[ok], np.where(pred_sev[ok] < 0, 0, pred_sev[ok]), labels=labels)
print(f"best model: {best_name} | pairs with a known severity: {ok.sum():,} of {len(ok):,}")
print("pair severity accuracy :", round(float((true_sev[ok] == np.where(pred_sev[ok] < 0, 0, pred_sev[ok])).mean()), 3))
print("pair severity macro-F1 :", round(float(f1_score(true_sev[ok], np.where(pred_sev[ok] < 0, 0, pred_sev[ok]), average="macro")), 3))
hi = true_sev == 2
print("High-severity pairs found (recall):", round(float((pred_sev[hi] == 2).mean()), 3) if hi.any() else "n/a")
display(pd.DataFrame(cm, index=["true Low", "true Medium", "true High"], columns=["pred Low", "pred Medium", "pred High"]))

## Step 9 · Try it: top predicted side effects for one pair

In [ ]:
def show_pair(name_a, name_b, model_name=best_name, top=8):
    names = drugs["drug_name"].str.lower().tolist()
    try:
        a, b = names.index(name_a.lower()), names.index(name_b.lower())
    except ValueError:
        print("drug not in the list; examples:", names[:10]); return
    X = np.hstack([c3.pair_fp_matrix(np.array([a]), np.array([b]), FP), CM[[a]] | CM[[b]], CM[[a]] & CM[[b]]]).astype(np.float32)
    if hist is not None:
        X = np.hstack([X, hist.transform(np.array([a]), np.array([b]), None)])
    s = c3.predict_scores(trained[model_name], X)[0]
    out = effects.assign(score=s).sort_values("score", ascending=False).head(top)
    display(out[["effect_name", "severity", "score"]].round(3))

show_pair("warfarin", "ibuprofen")

## Step 10 · Save the best model

In [ ]:
joblib.dump({"model": trained[best_name], "model_name": best_name, "threshold": thr, "effects": effects,
             "drugs": drugs[["drug_id", "drug_name", "drug_class", "main_use"]], "FP": FP, "CM": CM,
             "history": hist, "split_mode": SPLIT_MODE},
            RESULTS / f"best_model_{SPLIT_MODE}.joblib")
print("saved:", RESULTS / f"best_model_{SPLIT_MODE}.joblib")
other = "unseen_drug" if SPLIT_MODE == "pair" else "pair"
print(f"\nNow change SPLIT_MODE to '{other}' (Step 1) and run the notebook again to get the second result.")

In [ ]:
save_all('after part E1')

## Part E2 · The same, for NEW drugs (unseen-drug split) - the honest test

In [ ]:
import sys, importlib.util, subprocess
from pathlib import Path

HERE = Path.cwd()                                   # the folder of this notebook (component3)
if not (HERE / "c3_utils.py").exists():
    if Path("/content").exists():                   # Google Colab: get the code from GitHub
        REPO_DIR = Path("/content/repo")
        if not (REPO_DIR / "component3" / "c3_utils.py").exists():
            subprocess.run(["rm", "-rf", str(REPO_DIR)])
            subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/sksivakajan/mac-doc.git", str(REPO_DIR)])
        HERE = REPO_DIR / "component3"
        import os; os.chdir(HERE)
        print("Colab: code taken from GitHub ->", HERE)
    else:                                           # Jupyter was started from another folder
        HERE = Path(r"D:\med\data set\component3")
sys.path.insert(0, str(HERE))

def need(*pairs):
    """Install the packages that are missing. pairs = (import name, pip name)."""
    missing = [pip for mod, pip in pairs if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", missing)
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as e:
            print("!! Could not install", missing, "->", e)

need(("pandas", "pandas"), ("numpy", "numpy"), ("matplotlib", "matplotlib"),
     ("sklearn", "scikit-learn"), ("xgboost", "xgboost"), ("joblib", "joblib"))

import os, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
import c3_utils as c3
warnings.filterwarnings("ignore")

WORK = HERE / "work"
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)
SEED = c3.SEED

## Step 1 · Settings
| Setting | Meaning |
|---|---|
| `SPLIT_MODE` | `"pair"` (random pairs) or `"unseen_drug"` (drugs held out). |
| `REUSE_SAVED` | `True` = load a saved model (same settings) instead of training again. Every trained model is saved in `results/`. |
| `XGB_DEVICE` | `"cpu"` or `"cuda"` (GPU, Colab only). |
| `TRAIN_CAP` | Use at most this many training pairs. Smaller = faster. Set `None` to use all (needs more RAM and hours of time). |
| `SVM_TRAIN_CAP` | The SVM is slow, so it uses fewer pairs. |
| `USE_DRUG_HISTORY` | Add "how often did each drug cause each side effect in training" as features. |
| `TUNE` | `True` tries 2 settings per model on a small sample and keeps the better one (slower). |

In [ ]:
SPLIT_MODE = "unseen_drug"            # "pair"  or  "unseen_drug"
TRAIN_CAP = 60_000
SVM_TRAIN_CAP = 30_000
USE_DRUG_HISTORY = True
TUNE = False
MODELS_TO_RUN = ["random_forest", "xgboost", "svm"]
SVM_KERNEL = "linear"          # "linear" (fast) or "rbf" (very slow; use SVM_TRAIN_CAP <= 10_000)
N_JOBS = os.cpu_count() or 1
XGB_DEVICE = "cuda" if __import__("shutil").which("nvidia-smi") else "cpu"             # "cuda" = use a GPU (Google Colab with a GPU runtime makes XGBoost much faster)
REUSE_SAVED = True            # a model trained before with the SAME settings is loaded from the results folder instead of trained again

PARAMS = {   # first entry is used when TUNE = False
    "random_forest": [dict(n_estimators=150, min_samples_leaf=5, max_features="sqrt"),
                      dict(n_estimators=150, min_samples_leaf=2, max_features="sqrt")],
    "xgboost":       [dict(n_estimators=150, max_depth=5, learning_rate=0.1),
                      dict(n_estimators=150, max_depth=7, learning_rate=0.05)],
    "svm":           [dict(C=0.05), dict(C=0.5)],
}
print("split:", SPLIT_MODE, "| cores:", N_JOBS)

## Step 2 · Load the data and the split

In [ ]:
pairs = pd.read_csv(WORK / "modeling_pairs.csv")
Y = np.load(WORK / "modeling_Y.npy")
effects = pd.read_csv(WORK / "modeling_effects.csv")
drugs = pd.read_csv(WORK / "drugs_final.csv", keep_default_na=False)
z = np.load(WORK / "drug_features.npz")
FP, CM = z["FP"], z["CM"]
sp = np.load(WORK / f"split_{SPLIT_MODE}.npz")
tr_all, va, te = sp["train"], sp["val"], sp["test"]
ia, ib = pairs["idx_a"].to_numpy(), pairs["idx_b"].to_numpy()

sev_rank = effects["severity"].map({"Low": 0, "Medium": 1, "High": 2}).fillna(-1).to_numpy().astype(int)
high_mask = sev_rank == 2
print(f"train {len(tr_all):,}   val {len(va):,}   test {len(te):,}   | side effects {Y.shape[1]} (High severity: {high_mask.sum()})")

rng = np.random.default_rng(SEED)
def cap(idx, n):
    return np.sort(rng.choice(idx, n, replace=False)) if (n and len(idx) > n) else idx
tr = cap(tr_all, TRAIN_CAP)
print(f"pairs used for training: {len(tr):,}")

## Step 3 · Build the features
For a pair (A, B) the input numbers are (the same for A+B and B+A):
* **Fingerprints:** `A OR B` and `A AND B` (2 × 1024 numbers).
* **Drug classes:** which classes either drug has, and which both have.
* **Drug history** (optional): for each side effect, how often drug A and drug B had it in **training pairs**. For a *training* row the pair's own answer is removed first (leave-one-out), so the model can not read its own label. Validation/test rows never take part in this.

In [ ]:
hist = c3.DrugHistory(n_drugs=len(drugs)).fit(ia[tr_all], ib[tr_all], Y[tr_all]) if USE_DRUG_HISTORY else None

def build_X(idx, is_train):
    a, b = ia[idx], ib[idx]
    parts = [c3.pair_fp_matrix(a, b, FP), CM[a] | CM[b], CM[a] & CM[b]]
    X = np.hstack(parts).astype(np.float32)
    if hist is not None:
        X = np.hstack([X, hist.transform(a, b, Y[idx] if is_train else None)])
    return X

t0 = time.time()
Xtr, Xva, Xte = build_X(tr, True), build_X(va, False), build_X(te, False)
Ytr, Yva, Yte = Y[tr], Y[va], Y[te]
print(f"features: {Xtr.shape[1]} per pair   | train {Xtr.shape}  val {Xva.shape}  test {Xte.shape}   ({time.time()-t0:.0f}s)")
print(f"memory: about {(Xtr.nbytes + Xva.nbytes + Xte.nbytes)/1e9:.1f} GB")

## Step 4 · Baselines (no machine learning)
* **prevalence** – every pair gets the same list: the most common side effects in training.
* **drug_history** – average of the two drugs' own history (needs no model, no fingerprints).

If a trained model can not beat these, it has not learned anything useful.

In [ ]:
results, val_scores, test_scores = [], {}, {}

def record(name, S_va, S_te, extra=None):
    thr = c3.best_threshold(Yva, S_va)                 # cut-off chosen on VALIDATION only
    rv = c3.evaluate(Yva, S_va, thr, high_mask)
    rt = c3.evaluate(Yte, S_te, thr, high_mask)
    row = {"model": name, "threshold": round(thr, 4), **{f"val_{k}": v for k, v in rv.items()},
           **{f"test_{k}": v for k, v in rt.items()}}
    if extra:
        row.update(extra)
    results.append(row)
    val_scores[name], test_scores[name] = S_va, S_te
    print(f"{name:16s} val micro-AUPRC {rv['micro_AUPRC']:.3f} | TEST micro-AUPRC {rt['micro_AUPRC']:.3f}  "
          f"P@3 {rt['P@3']:.3f}  P@10 {rt['P@10']:.3f}  micro-F1 {rt['micro_F1']:.3f}  High-recall {rt.get('recall_High_effects', float('nan')):.3f}")

base = c3.PrevalenceBaseline().fit(Xtr, Y[tr_all])
record("prevalence", base.predict_proba(Xva), base.predict_proba(Xte))

if hist is not None:
    K = Y.shape[1]
    def hist_score(idx):
        h = hist.transform(ia[idx], ib[idx], None)
        return (h[:, :K] + h[:, K:2 * K]) / 2
    record("drug_history", hist_score(va), hist_score(te))

## Step 5 · Train the three models
Time is printed for each one. Run this cell once; the models stay in memory for the next steps.

In [ ]:
def make_model(name, p):
    from sklearn.multiclass import OneVsRestClassifier
    if name == "random_forest":
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(n_jobs=N_JOBS, random_state=SEED, **p)
    if name == "xgboost":
        from xgboost import XGBClassifier
        return OneVsRestClassifier(XGBClassifier(tree_method="hist", device=XGB_DEVICE, subsample=0.8, colsample_bytree=0.5,
                                                 n_jobs=N_JOBS, eval_metric="logloss", random_state=SEED,
                                                 verbosity=0, **p))
    if name == "svm":
        if SVM_KERNEL == "linear":
            from sklearn.svm import LinearSVC
            base = LinearSVC(dual=False, random_state=SEED, **p)
        else:
            from sklearn.svm import SVC
            base = SVC(kernel="rbf", gamma="scale", **p)
        return OneVsRestClassifier(base, n_jobs=N_JOBS)
    raise ValueError(name)

trained = {}
for name in MODELS_TO_RUN:
    n_fit = min(len(Xtr), SVM_TRAIN_CAP) if name == "svm" else len(Xtr)
    Xf, Yf = Xtr[:n_fit], Ytr[:n_fit]
    params = PARAMS[name][0]
    t0 = time.time()
    if TUNE:                                   # pick the better setting on VALIDATION, using a small sample
        n_small = min(15_000, n_fit)
        best_p, best_s = None, -1
        for p in PARAMS[name]:
            m = make_model(name, p).fit(Xf[:n_small], Yf[:n_small])
            s = c3.evaluate(Yva, c3.predict_scores(m, Xva), 0.5)["micro_AUPRC"]
            print(f"   {name} {p} -> val micro-AUPRC {s:.3f}")
            if s > best_s:
                best_p, best_s = p, s
        params = best_p
    sig = dict(name=name, params=str(params), n_fit=int(n_fit), mode=SPLIT_MODE, drug_history=bool(USE_DRUG_HISTORY),
               svm_kernel=SVM_KERNEL if name == "svm" else "", n_features=int(Xf.shape[1]))
    mfile = RESULTS / f"model_{SPLIT_MODE}_{name}.joblib"
    model, secs = None, 0.0
    if REUSE_SAVED and mfile.exists():
        saved = joblib.load(mfile)
        if saved.get("sig") == sig:
            model, secs = saved["model"], saved["seconds"]
            print(f"   loaded the saved {name} model ({mfile.name}); it was trained in {secs/60:.1f} min")
    if model is None:
        model = make_model(name, params).fit(Xf, Yf)
        secs = time.time() - t0
        joblib.dump({"model": model, "sig": sig, "seconds": secs}, mfile, compress=3)   # keep it: no need to train again
    trained[name] = model
    record(name, c3.predict_scores(model, Xva), c3.predict_scores(model, Xte),
           {"params": str(params), "train_pairs": n_fit, "train_seconds": round(secs)})
    print(f"   -> trained in {secs/60:.1f} min\n")

## Step 6 · Results table

In [ ]:
res = pd.DataFrame(results)
cols = ["model", "test_micro_AUPRC", "test_macro_AUPRC", "test_P@3", "test_P@5", "test_P@10", "test_R@10",
        "test_micro_F1", "test_macro_F1", "test_recall_High_effects", "val_micro_AUPRC"]
show = res[cols].round(3)
display(show)
res.to_csv(RESULTS / f"metrics_{SPLIT_MODE}.csv", index=False)

learned = res[~res["model"].isin(["prevalence", "drug_history"])]
best_name = learned.sort_values("val_micro_AUPRC", ascending=False)["model"].iloc[0]   # chosen on VALIDATION
print("Best trained model (by validation micro-AUPRC):", best_name)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, m, t in zip(ax, ["test_micro_AUPRC", "test_P@3", "test_micro_F1"], ["micro-AUPRC", "Precision@3", "micro-F1"]):
    a.bar(res["model"], res[m], color=["#999" if x in ("prevalence", "drug_history") else "#3b6ea5" for x in res["model"]])
    a.set_title(t + " (test)"); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(RESULTS / f"comparison_{SPLIT_MODE}.png", dpi=130); plt.show()

## Step 7 · Sanity checks (do this before you trust the numbers)
1. **Overfitting gap:** training score far above test score means the model memorised.
2. **Label-shuffle test:** we shuffle the training answers and train again. A healthy pipeline must then fall back to the *prevalence* baseline. If it stays clearly **above** the baseline, labels are leaking into the features.

In [ ]:
# 1. overfitting gap on a sample of training rows
sample = np.arange(min(8000, len(Xtr)))
rows = []
for name, m in trained.items():
    s_tr = c3.evaluate(Ytr[sample], c3.predict_scores(m, Xtr[sample]), 0.5)["micro_AUPRC"]
    s_te = res.loc[res["model"] == name, "test_micro_AUPRC"].iloc[0]
    rows.append({"model": name, "train_micro_AUPRC": round(s_tr, 3), "test_micro_AUPRC": round(s_te, 3), "gap": round(s_tr - s_te, 3)})
display(pd.DataFrame(rows))
print("Note: a gap is normal for Random Forest (it sees its own training rows). Look at the TEST column for quality.")

# 2. label-shuffle test (fast Random Forest on a small sample)
from sklearn.ensemble import RandomForestClassifier
Ys = Y[tr_all].copy()
np.random.default_rng(1).shuffle(Ys, axis=0)                    # break the link between pairs and answers
hist_s = c3.DrugHistory(len(drugs)).fit(ia[tr_all], ib[tr_all], Ys) if USE_DRUG_HISTORY else None
sub = tr[:15_000]
pos = {int(p): i for i, p in enumerate(tr_all)}
Ys_sub = Ys[[pos[int(p)] for p in sub]]
def X_shuf(idx, own=None):
    a, b = ia[idx], ib[idx]
    X = np.hstack([c3.pair_fp_matrix(a, b, FP), CM[a] | CM[b], CM[a] & CM[b]]).astype(np.float32)
    return np.hstack([X, hist_s.transform(a, b, own)]) if hist_s is not None else X
rf = RandomForestClassifier(n_estimators=60, min_samples_leaf=5, n_jobs=N_JOBS, random_state=SEED).fit(X_shuf(sub, Ys_sub), Ys_sub)
shuf = c3.evaluate(Yte, c3.predict_scores(rf, X_shuf(te)), 0.5)["micro_AUPRC"]
basel = res.loc[res["model"] == "prevalence", "test_micro_AUPRC"].iloc[0]
print(f"label-shuffle model: test micro-AUPRC {shuf:.3f}   |   prevalence baseline {basel:.3f}")
print("PASS: no sign of leakage" if shuf < basel + 0.02 else "!! WARNING: shuffled model beats the baseline -> look for leakage")

## Step 8 · Severity (Low / Medium / High) for a pair
A pair gets the **worst severity among its side effects**. We compare the true severity (from the TWOSIDES signals) with the severity from the predicted side effects.

⚠ The severity of each side effect comes from the placeholder rules in `severity.py`. Agree a real source with your supervisor before using these numbers in the thesis.

⚠ About 28 of the side effects are marked High, so **most pairs contain at least one High effect**. "Worst severity" is therefore dominated by the High class. Talk with your supervisor about a better pair-level rule, for example *"High only if the predicted chance of a High effect is above a higher cut-off"*, or a severity **score** (the sum of the probabilities of High effects).

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score
thr = float(res.loc[res["model"] == best_name, "threshold"].iloc[0])
S_best = test_scores[best_name]
true_sev = c3.pair_severity(Yte, sev_rank)
pred_sev = c3.pair_severity(S_best >= thr, sev_rank)
ok = true_sev >= 0
labels = [0, 1, 2]
cm = confusion_matrix(true_sev[ok], np.where(pred_sev[ok] < 0, 0, pred_sev[ok]), labels=labels)
print(f"best model: {best_name} | pairs with a known severity: {ok.sum():,} of {len(ok):,}")
print("pair severity accuracy :", round(float((true_sev[ok] == np.where(pred_sev[ok] < 0, 0, pred_sev[ok])).mean()), 3))
print("pair severity macro-F1 :", round(float(f1_score(true_sev[ok], np.where(pred_sev[ok] < 0, 0, pred_sev[ok]), average="macro")), 3))
hi = true_sev == 2
print("High-severity pairs found (recall):", round(float((pred_sev[hi] == 2).mean()), 3) if hi.any() else "n/a")
display(pd.DataFrame(cm, index=["true Low", "true Medium", "true High"], columns=["pred Low", "pred Medium", "pred High"]))

## Step 9 · Try it: top predicted side effects for one pair

In [ ]:
def show_pair(name_a, name_b, model_name=best_name, top=8):
    names = drugs["drug_name"].str.lower().tolist()
    try:
        a, b = names.index(name_a.lower()), names.index(name_b.lower())
    except ValueError:
        print("drug not in the list; examples:", names[:10]); return
    X = np.hstack([c3.pair_fp_matrix(np.array([a]), np.array([b]), FP), CM[[a]] | CM[[b]], CM[[a]] & CM[[b]]]).astype(np.float32)
    if hist is not None:
        X = np.hstack([X, hist.transform(np.array([a]), np.array([b]), None)])
    s = c3.predict_scores(trained[model_name], X)[0]
    out = effects.assign(score=s).sort_values("score", ascending=False).head(top)
    display(out[["effect_name", "severity", "score"]].round(3))

show_pair("warfarin", "ibuprofen")

## Step 10 · Save the best model

In [ ]:
joblib.dump({"model": trained[best_name], "model_name": best_name, "threshold": thr, "effects": effects,
             "drugs": drugs[["drug_id", "drug_name", "drug_class", "main_use"]], "FP": FP, "CM": CM,
             "history": hist, "split_mode": SPLIT_MODE},
            RESULTS / f"best_model_{SPLIT_MODE}.joblib")
print("saved:", RESULTS / f"best_model_{SPLIT_MODE}.joblib")
other = "unseen_drug" if SPLIT_MODE == "pair" else "pair"
print(f"\nNow change SPLIT_MODE to '{other}' (Step 1) and run the notebook again to get the second result.")

## Step 11 · Both splits side by side (after you ran both)

In [ ]:
parts = []
for mode in ["pair", "unseen_drug"]:
    f = RESULTS / f"metrics_{mode}.csv"
    if f.exists():
        m = pd.read_csv(f)[["model", "test_micro_AUPRC", "test_P@3", "test_P@10", "test_micro_F1", "test_recall_High_effects"]]
        m.insert(0, "split", mode)
        parts.append(m)
if parts:
    both = pd.concat(parts, ignore_index=True).round(3)
    display(both)
    both.to_csv(RESULTS / "metrics_both_splits.csv", index=False)
else:
    print("No saved results yet.")

In [ ]:
save_all('after part E2')

## Part F · Save the results

In [ ]:
#@title F1 · Zip the results and download them
import shutil, os
from google.colab import files
os.makedirs("results", exist_ok=True)
small = ["results", "work/modeling_effects.csv", "work/drugs_final.csv", "work/drugs_without_smiles.csv"]
os.makedirs("/content/out", exist_ok=True)
for p in small:
    if os.path.exists(p):
        dst = os.path.join("/content/out", p)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        (shutil.copytree if os.path.isdir(p) else shutil.copy)(p, dst, **({"dirs_exist_ok": True} if os.path.isdir(p) else {}))
shutil.make_archive("/content/c3_results", "zip", "/content/out")
print("zip size (MB):", round(os.path.getsize("/content/c3_results.zip") / 1e6, 1))
files.download("/content/c3_results.zip")

In [ ]:
#@title F2 · (Optional) Push the small result files back to GitHub  -- set PUSH = True to run
PUSH = False
if PUSH:
    assert TOKEN, "Pushing needs the GITHUB_TOKEN secret with write access to this repo."
    run(["git", "config", "user.email", "colab@example.com"])
    run(["git", "config", "user.name", "MedSafe Colab"])
    run(["git", "checkout", "-B", "colab-results"])
    os.makedirs("outputs", exist_ok=True)
    for f in os.listdir("results"):
        if f.endswith((".csv", ".png")):
            shutil.copy(os.path.join("results", f), os.path.join("outputs", f))
    run(["git", "add", "outputs", "work/modeling_effects.csv", "work/drugs_final.csv", "work/drug_fp_colab.npz"])
    print(run(["git", "commit", "-m", "Colab results: metrics and fingerprints"]))
    run(["git", "push", "-f", url, "colab-results"])
    print("Pushed to branch  colab-results  (open GitHub and make a Pull Request if you want it in main).")
else:
    print("Not pushing (PUSH = False).")

## How to read your results
* Compare each model with **prevalence** and **drug_history**. A real improvement is a higher `test_micro_AUPRC` and `P@3/P@10`.
* The `unseen_drug` result is lower; that is normal and honest: it shows how the system behaves for a **new** medicine.
* `recall_High_effects` is the share of serious side effects the model finds; for safety it matters more than overall F1.
* TWOSIDES comes from reports, so even a perfect model learns *reported signals*, not proven causes.